# Step 1: Create your first AgentScope agent

In AgentScope 2.x, use `Agent` to create an agent. This notebook creates one agent with no tools, so it produces one response. The next lesson adds a tool and begins the ReAct cycle.

In [ ]:
import os

from dotenv import load_dotenv

from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel

## Step 2: Configure the model connection

![First AgentScope agent workflow](figures/first-agentscope-agent-workflow.svg)

The agent receives one user message, uses its instructions and language model, and returns one answer. No tools are available yet.

In [ ]:
load_dotenv()

model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")
if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=220),
)

## Step 3: Create the text-only agent

AgentScope looks at what the model returns. If the model asks to use a tool, AgentScope runs the tool, adds the result to the conversation, and asks the model again. If the model returns normal text and does not ask for a tool, that text is the final answer. This lesson has no tools, so the first response should be the final answer.

`ReActConfig(max_iters=3)` is a safety limit for later lessons with tools. It stops the agent after three tool-use tries.

In [ ]:
triage_agent = Agent(
    name="triage_assistant",
    system_prompt=(
        "You help review notes about computer activity. "
        "Separate what the note says from what it might mean. "
        "When evidence is insufficient, say so plainly."
    ),
    model=model,
    react_config=ReActConfig(max_iters=3),
)

print(f"Created: {triage_agent.name}")

## Step 4: Ask the agent to review a case note

The next cell sends a fictional case note to the agent and prints its response. With no tools registered, this one text response is the complete run.

In [ ]:
case_note = """
At 09:14 UTC, an employee's work computer contacted 198.51.100.23 forty-three times.
The available network record does not show what was sent or which program made the contacts.
"""

# AgentScope sends this message to the language model through triage_agent.
# With no tools registered, a text-only response is the final answer.
response = await triage_agent.reply(
    Msg(
        name="analyst",
        role="user",
        content=[
            TextBlock(
                text=(
                    "Review this case note. Return two short sections: "
                    "What the note says and What it might mean.\n\n"
                    f"{case_note.strip()}"
                ),
            ),
        ],
    ),
)

# The response can contain several parts; print only its text.
print("".join(block.text for block in response.content if isinstance(block, TextBlock)))